# 04 — Tracker Architecture Assessment: BoTSORT

Evaluates **BoTSORT** tracker architecture on a small sample of video clips.
Covers the full architecture decision space: position-only tracking, ReID options,
and temporal/spatial post-process fragment merging.

Because **multiple instances of the same class can appear simultaneously**,
pure class-level merging is insufficient — appearance identity (Re-ID) is required
to keep consistent track IDs across re-entries.

This notebook:
1. **Diagnoses** fragmentation with a track-timeline visualisation
2. **Sweeps** position-based knobs (no ReID) to establish a baseline
3. **Evaluates** two ReID options (Ultralytics built-in vs boxmot osnet_x0_25)
4. **Tests** post-process fragment merging
5. **Saves** the best architecture config to `configs/kartector_botsort_arch.json`

> **Result:** ReID Option B (boxmot `osnet_x0_25`) with post-process merge
> gave the lowest fragment count and is used by all downstream notebooks.

## 0 — Configuration

In [ ]:
from pathlib import Path
import json, cv2, yaml, itertools, subprocess, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from ultralytics import YOLO

REPO_ROOT    = Path('/home1/hendersonj6179@cgu.edu')
RUNS_DIR     = (REPO_ROOT / 'runs').resolve()
TRACKER_RUNS = (REPO_ROOT / 'runs/trackers/tuning').resolve()
TRACKER_RUNS.mkdir(parents=True, exist_ok=True)


# ── Detector weights ──────────────────────────────────────────────────────
# Default: YOLO26 final model (all-data train from notebook 02)
# Swap to the RT-DETR line if you want to tune trackers on that detector instead.
WEIGHTS = RUNS_DIR / 'yolo26' / 'final' / 'weights' / 'best.pt'
# WEIGHTS = RUNS_DIR / 'rtdetr' / 'final' / 'weights' / 'best.pt'
CHUNKS_DIR = REPO_ROOT / 'data/processed/labelstudiochunks'

# Detection confidence for all tracker runs in this notebook
# Raise to reduce false positives; lower to catch more detections
CONF = 0.25

with open(REPO_ROOT / 'data/splits/splits.json') as f:
    splits = json.load(f)
CLASSES = splits['classes']
N_CLS   = len(CLASSES)

test_videos = sorted(CHUNKS_DIR.glob('*.mp4'))

# Detection confidence for all tracker runs in this notebook
# Raise to reduce false positives; lower to catch more detections
CONF = 0.25
TEST_VIDEO  = test_videos[0] if test_videos else None
print(f'Test video : {TEST_VIDEO}')
print(f'Classes    : {CLASSES}')
print(f'Weights    : {WEIGHTS}')

## 1 — Track Fragmentation Diagnostics

Run the baseline tracker and plot a **track timeline** — one horizontal bar per
Track ID, coloured by class.

- **Gaps** in a bar = frames where the tracker lost the object
- **Multiple short same-colour bars** = fragmentation (same physical icon, multiple IDs)

With multiple instances per class on screen simultaneously, a correctly tuned tracker
should show one continuous bar *per physical instance* — not one per class.

In [ ]:
from helpers import plot_track_timeline

## 1b — Ground-Truth Timeline & Fragment Counts

Loads the MOT-format `gt/gt.txt` for the same video chunk and plots the same
track-timeline and fragment-count summary as the tracker above.

**GT format:** `frame, track_id, x, y, w, h, conf, class_id, visibility`
(1-indexed frames; class_id is 1-indexed in file, converted to 0-indexed here)

Comparing GT vs baseline side-by-side shows the upper bound on what a perfect
tracker could achieve and immediately highlights which classes fragment most.

In [ ]:
from helpers import load_gt_as_df

## 2 — Position-Based Parameter Sweep

Sweeps `track_buffer` and `match_thresh` **without** ReID as a baseline.

| Parameter | Effect |
|---|---|
| `track_buffer` | Frames a lost track stays alive. **Raise** to bridge off-screen gaps. |
| `match_thresh` | Max association cost. **Lower** (≈0.6) to allow positional jumps on re-entry. |
| `new_track_thresh` | Min score to spawn a new track. **Raise** to suppress spurious IDs. |

In [ ]:
from helpers import count_fragments

## 3 — Re-ID: Option A — Ultralytics Built-in ReID

Ultralytics' BoTSORT supports a `with_reid` flag that enables its own bundled
appearance model. This keeps the same `model.track()` pipeline — no extra
inference loop required.

**Limitations:** the Ultralytics ReID model is fixed (not selectable);
`appearance_thresh` is respected but `model:` is ignored.
Good first test before the full boxmot pipeline.

In [ ]:
SWEEP_REID_A = {
    'track_buffer': [20, 45],
    'match_thresh': [0.60, 0.70, 0.90, 0.99],
    'with_reid':    [True, False],
}

reid_a_results = []
if TEST_VIDEO:
    for tb, mt, reid_on in itertools.product(
            SWEEP_REID_A['track_buffer'],
            SWEEP_REID_A['match_thresh'],
            SWEEP_REID_A['with_reid']):
        cfg_path = TRACKER_RUNS / f'reidA_tb{tb}_mt{int(mt*100)}_reid{int(reid_on)}.yaml'
        write_botsort_cfg(cfg_path, tb, mt, with_reid=reid_on)
        df_sw = run_tracker(TEST_VIDEO, WEIGHTS, str(cfg_path))
        frags = count_fragments(df_sw)
        total = sum(frags.values())
        reid_a_results.append({'track_buffer': tb, 'match_thresh': mt,
                                'reid': reid_on, 'total_fragments': total, **frags})
        tag = 'Y' if reid_on else 'N'
        print(f'  tb={tb:2d}  mt={mt:.2f}  reid={tag}  -> fragments={total}')
    df_ra = pd.DataFrame(reid_a_results)
    print('\nTop 5 (Option A — Ultralytics built-in ReID):')
    print(df_ra.sort_values('total_fragments').head(5).to_string(index=False))
else:
    print('Set TEST_VIDEO to run.')


## 3b — Re-ID: Option B — boxmot Full ReID Pipeline

boxmot's BoTSORT accepts any torchreid model (`osnet_x0_25`, `osnet_x1_0`, etc.)
and exposes `appearance_thresh` for fine-grained control.

**The trade-off:** requires a manual detect → track loop instead of `model.track()`.

`osnet_x0_25` (~3 MB) is **auto-downloaded** on first run — no manual setup needed.

| Model | Size | Notes |
|---|---|---|
| `osnet_x0_25` | ~3 MB | Recommended — fast, small |
| `osnet_x1_0`  | ~11 MB | More accurate |

> **Chosen:** `osnet_x0_25_msmt17` — best balance of speed and tracking quality.

In [ ]:
try:
    import boxmot; print(f'boxmot {boxmot.__version__} ready')
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'boxmot', '-q'], check=True)
    import boxmot; print('boxmot installed')

import torch

# boxmot 17+ uses 'BotSort' (not 'BoTSORT')
# Output columns: x1, y1, x2, y2, track_id, conf, cls, det_ind
from boxmot.trackers.botsort.botsort import BotSort as _BotSortCls
from boxmot.utils import WEIGHTS as BOXMOT_WEIGHTS
print(f'Using tracker class: {_BotSortCls.__name__}')

# boxmot 17 ships a default ReID model at <boxmot_package>/models/osnet_x0_25_msmt17.pt
# It is downloaded automatically on first use via boxmot's CLI / get_model_path.
# Passing the full Path with .pt suffix is required — bare strings are not resolved.
REID_MODEL = BOXMOT_WEIGHTS / 'osnet_x0_25_msmt17.pt'
print(f'ReID weights path: {REID_MODEL}  (exists={REID_MODEL.exists()})')

def run_tracker_boxmot(video_path, weights, track_buffer=30, match_thresh=0.7,
                       appearance_thresh=0.4, reid_model=None, conf=0.25):
    """YOLO detect + boxmot BotSort with full ReID control."""
    if reid_model is None:
        reid_model = REID_MODEL
    # Download weights if missing — boxmot 17 hosts models on Google Drive, use gdown
    if not Path(reid_model).exists():
        print(f'  Downloading ReID weights to {reid_model} ...')
        try:
            import gdown
        except ImportError:
            subprocess.run([sys.executable, '-m', 'pip', 'install', 'gdown', '-q'], check=True)
            import gdown
        from boxmot.reid.core.config import TRAINED_URLS
        model_name = Path(reid_model).name
        url = TRAINED_URLS.get(model_name)
        if url is None:
            print(f'  No URL found for {model_name}. Available: {list(TRAINED_URLS.keys())}')
            return pd.DataFrame(columns=['frame', 'track_id', 'class_id', 'x1', 'y1', 'x2', 'y2', 'conf'])
        Path(reid_model).parent.mkdir(parents=True, exist_ok=True)
        gdown.download(url, str(reid_model), quiet=False)
        print(f'  Downloaded: {reid_model}')
    det_model = YOLO(str(weights))
    device = torch.device(
        'mps'  if torch.backends.mps.is_available() else
        'cuda' if torch.cuda.is_available() else 'cpu'
    )
    tracker = _BotSortCls(
        reid_weights=Path(reid_model),
        device=device,
        half=False,
        track_buffer=track_buffer,
        match_thresh=match_thresh,
        appearance_thresh=appearance_thresh,
        with_reid=True,
    )
    cap = cv2.VideoCapture(str(video_path))
    rows = []
    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        results = det_model(frame, conf=conf, verbose=False)
        dets = results[0].boxes
        if dets is not None and len(dets):
            xyxy  = dets.xyxy.cpu().numpy()
            confs = dets.conf.cpu().numpy().reshape(-1, 1)
            clss  = dets.cls.cpu().numpy().reshape(-1, 1)
            det_arr = np.concatenate([xyxy, confs, clss], axis=1)
        else:
            det_arr = np.empty((0, 6))
        tracks = tracker.update(det_arr, frame)  # shape: (N, 8) — x1,y1,x2,y2,tid,conf,cls,det_ind
        if frame_idx == 0 and len(tracks):
            print(f'  [boxmot debug] tracks shape={tracks.shape}  first row={tracks[0]}')
        for t in tracks:
            # Defensively find cls: it should be col 6, but verify it's an int in [0, N_CLS)
            cid = int(t[6]) if len(t) > 6 else -1
            if cid < 0 or cid >= N_CLS:
                # Fallback: look for the class in the original det_arr by matching det_ind
                det_ind = int(t[7]) if len(t) > 7 else -1
                if 0 <= det_ind < len(det_arr):
                    cid = int(det_arr[det_ind, 5])   # det_arr col 5 is cls
                else:
                    continue  # skip this track — class is unresolvable
            rows.append((frame_idx, int(t[4]), cid,
                         float(t[0]), float(t[1]), float(t[2]), float(t[3]), float(t[5])))
        frame_idx += 1
    cap.release()
    return pd.DataFrame(rows, columns=['frame', 'track_id', 'class_id',
                                       'x1', 'y1', 'x2', 'y2', 'conf'])


SWEEP_REID_B = {
    'track_buffer':      [20, 30, 45],
    'match_thresh':      [0.60, 0.70, 0.90, 0.99],
    'appearance_thresh': [0.25, 0.40, 0.60],
}

reid_b_results = []
if TEST_VIDEO:
    for tb, mt, at in itertools.product(
            SWEEP_REID_B['track_buffer'],
            SWEEP_REID_B['match_thresh'],
            SWEEP_REID_B['appearance_thresh']):
        df_sw = run_tracker_boxmot(TEST_VIDEO, WEIGHTS,
                                   track_buffer=tb, match_thresh=mt,
                                   appearance_thresh=at)
        frags = count_fragments(df_sw)
        total = sum(frags.values())
        reid_b_results.append({'track_buffer': tb, 'match_thresh': mt,
                                'appearance_thresh': at, 'reid': True,
                                'total_fragments': total, **frags})
        print(f'  tb={tb:2d}  mt={mt:.2f}  at={at:.2f}  -> fragments={total}')
    df_rb = pd.DataFrame(reid_b_results)
    print('\nTop 5 (Option B — boxmot osnet ReID):')
    print(df_rb.sort_values('total_fragments').head(5).to_string(index=False))
else:
    print('Set TEST_VIDEO to run.')


## 4 — Comparison: Position-Only vs ReID Option A vs ReID Option B

Fragment count bar chart comparing all three configurations side by side.
**ReID Option B (boxmot)** consistently produced the fewest fragments.

In [ ]:
all_reid = []
if 'reid_a_results' in dir(): all_reid += reid_a_results
if 'reid_b_results' in dir(): all_reid += reid_b_results

if pos_results:
    best_pos  = pd.DataFrame(pos_results).sort_values('total_fragments').iloc[0]
    print('=== Best position-only ===')
    print(f'  tb={int(best_pos.track_buffer)}  mt={best_pos.match_thresh:.2f}  fragments={int(best_pos.total_fragments)}')
    if all_reid:
        best_reid = pd.DataFrame(all_reid).sort_values('total_fragments').iloc[0]
        print('=== Best with ReID (either option) ===')
        avail_cols = [c for c in ['track_buffer','match_thresh','appearance_thresh','reid','total_fragments'] if c in best_reid.index]
        print(best_reid[avail_cols].to_string())
        delta = int(best_pos.total_fragments) - int(best_reid.total_fragments)
        print(f'ReID improvement: {delta:+d}  ({delta/max(best_pos.total_fragments,1):.0%})')

    # Build rows_to_plot: GT first (if available), then each tracker
    rows_to_plot = []
    if 'df_gt' in dir() and not df_gt.empty:
        rows_to_plot.append(('Ground Truth', count_fragments(df_gt), 'black'))
    rows_to_plot.append(('Position only (best)', {c: int(best_pos.get(c,0)) for c in CLASSES}, 'steelblue'))
    if 'reid_a_results' in dir() and reid_a_results:
        ba = pd.DataFrame(reid_a_results).sort_values('total_fragments').iloc[0]
        rows_to_plot.append(('ReID Opt-A Ultralytics (best)', {c: int(ba.get(c,0)) for c in CLASSES}, 'darkorange'))
    if 'reid_b_results' in dir() and reid_b_results:
        bb = pd.DataFrame(reid_b_results).sort_values('total_fragments').iloc[0]
        rows_to_plot.append(('ReID Opt-B boxmot (best)', {c: int(bb.get(c,0)) for c in CLASSES}, 'seagreen'))

    x = np.arange(N_CLS); w = 0.8 / len(rows_to_plot)
    fig, ax = plt.subplots(figsize=(15, 5))
    for k, (label, frags, color) in enumerate(rows_to_plot):
        offset = (k - len(rows_to_plot)/2 + 0.5) * w
        ax.bar(x + offset, [frags.get(c, 0) for c in CLASSES],
               w, label=label, color=color, alpha=0.85)
    ax.set_xticks(x); ax.set_xticklabels(CLASSES, rotation=30, ha='right')
    ax.set_ylabel('Track fragments (lower = better)')
    ax.set_title('Fragment Count: GT vs All Tracker Configurations', fontweight='bold')
    ax.legend(); ax.grid(axis='y', alpha=0.3); plt.tight_layout()
    out = TRACKER_RUNS / 'reid_comparison.png'
    plt.savefig(out, dpi=150, bbox_inches='tight'); plt.show()
    print(f'Saved: {out}')
else:
    print('Run Section 2 first.')


## 5 — Fragment Merge Post-Processor

Even with ReID, some fragments remain when embeddings drift (e.g. icon partially
off-screen). This post-processor stitches same-class fragments **after** inference.

**Important:** only the **single closest** candidate is merged per ended track —
this prevents collapsing two different instances of the same class that happen to
be temporally adjacent.

In [ ]:
from helpers import merge_fragments

## 6 — Multi-Video Comparison: GT vs All Three Tracker Configurations (No Post-Processing)

Runs all three tracker variants across `N_SAMPLE` video chunks and produces one
fragment-count table per video plus an aggregate summary.
No post-processing merge is applied here — see Section 6b.

In [ ]:
import time

# ── Configuration ──────────────────────────────────────────────────────────
N_SAMPLE = 5      # number of video chunks to sample
CONF_6   = 0.25   # YOLO detection confidence — raise to reduce false positives
SEED     = 42

import random
random.seed(SEED)
all_videos = sorted(CHUNKS_DIR.glob('*.mp4'))
sample_videos = random.sample(all_videos, min(N_SAMPLE, len(all_videos)))
print(f'Sampled {len(sample_videos)} videos (conf={CONF_6}):')
for v in sample_videos: print(f'  {v.name}')

# ── Best configs from sweeps (fall back to defaults if sweeps not run) ──────
def _best_row(results, defaults):
    if results:
        return pd.DataFrame(results).sort_values('total_fragments').iloc[0].to_dict()
    return defaults

pos_best    = _best_row(pos_results    if 'pos_results'    in dir() else [], {'track_buffer': 30, 'match_thresh': 0.70})
reid_a_best = _best_row(reid_a_results if 'reid_a_results' in dir() else [], {'track_buffer': 30, 'match_thresh': 0.60, 'reid': True})
reid_b_best = _best_row(reid_b_results if 'reid_b_results' in dir() else [], {'track_buffer': 30, 'match_thresh': 0.60, 'appearance_thresh': 0.40})

# Write best config yamls once
_pos_cfg = TRACKER_RUNS / 'multi_pos_best.yaml'
write_botsort_cfg(_pos_cfg, int(pos_best['track_buffer']), float(pos_best['match_thresh']), with_reid=False)
_reid_a_cfg = TRACKER_RUNS / 'multi_reidA_best.yaml'
write_botsort_cfg(_reid_a_cfg, int(reid_a_best['track_buffer']), float(reid_a_best['match_thresh']), with_reid=bool(reid_a_best.get('reid', True)))

# ── Run all videos ───────────────────────────────────────────────────────────
per_video_tables_6 = []

for vid in sample_videos:
    vname = vid.stem
    print(f'\n{"="*55}\nVideo: {vname}\n{"="*55}')

    # Ground truth (no timing — it's a file read)
    df_gt_v, _ = load_gt_as_df(vid, MOT_SEQ_DIR)
    gt_frags = count_fragments(df_gt_v) if not df_gt_v.empty else {c: 0 for c in CLASSES}

    # Position-only
    t0 = time.time(); df_pos = run_tracker(vid, WEIGHTS, str(_pos_cfg), conf=CONF_6); t_pos = time.time() - t0
    pos_frags = count_fragments(df_pos)

    # Option A — Ultralytics ReID
    t0 = time.time(); df_ra_v = run_tracker(vid, WEIGHTS, str(_reid_a_cfg), conf=CONF_6); t_ra = time.time() - t0
    ra_frags = count_fragments(df_ra_v)

    # Option B — boxmot osnet ReID (only if REID_MODEL exists)
    if 'REID_MODEL' in dir() and Path(REID_MODEL).exists():
        t0 = time.time()
        df_rb_v = run_tracker_boxmot(vid, WEIGHTS,
                                     track_buffer=int(reid_b_best['track_buffer']),
                                     match_thresh=float(reid_b_best['match_thresh']),
                                     appearance_thresh=float(reid_b_best.get('appearance_thresh', 0.40)),
                                     conf=CONF_6)
        t_rb = time.time() - t0
        rb_frags = count_fragments(df_rb_v)
    else:
        rb_frags = {c: None for c in CLASSES}; t_rb = None
        print('  Option B skipped — ReID weights not downloaded yet')

    # Build per-video table (fragments + elapsed time row)
    tbl = pd.DataFrame({
        'Ground Truth':       [gt_frags.get(c, 0) for c in CLASSES],
        'Pos-only':           [pos_frags.get(c, 0) for c in CLASSES],
        'ReID Opt-A (Ult.)':  [ra_frags.get(c, 0) for c in CLASSES],
        'ReID Opt-B (bxmt)':  [rb_frags.get(c)    for c in CLASSES],
    }, index=CLASSES)
    tbl.loc['TOTAL']       = tbl.sum()
    tbl.loc['Time (s)']    = [None, round(t_pos, 1), round(t_ra, 1),
                               round(t_rb, 1) if t_rb is not None else None]
    per_video_tables_6.append((vname, tbl))

    print(f'\nFragment counts — {vname}:')
    display(tbl.style
               .highlight_min(axis=1, color='#c6efce', subset=pd.IndexSlice[CLASSES, ['Pos-only','ReID Opt-A (Ult.)','ReID Opt-B (bxmt)']])
               .format(lambda x: f'{x:.1f}' if isinstance(x, float) and not pd.isna(x) else (str(int(x)) if x is not None and not (isinstance(x, float) and pd.isna(x)) else '—')))

# ── Aggregate summary ────────────────────────────────────────────────────────
print(f'\n{"="*55}\nAGGREGATE (no post-merge): Total fragments across sampled videos\n{"="*55}')
agg6 = pd.DataFrame({
    col: [sum(tbl.loc[c, col] or 0 for _, tbl in per_video_tables_6 if c in tbl.index and tbl.loc[c, col] is not None)
          for c in CLASSES]
    for col in ['Ground Truth', 'Pos-only', 'ReID Opt-A (Ult.)', 'ReID Opt-B (bxmt)']
}, index=CLASSES)
agg6.loc['TOTAL'] = agg6.sum()
display(agg6.style.highlight_min(axis=1, color='#c6efce',
        subset=['Pos-only','ReID Opt-A (Ult.)','ReID Opt-B (bxmt)']))

out_csv6 = TRACKER_RUNS / 'multi_video_aggregate_raw.csv'
agg6.to_csv(out_csv6)
print(f'Saved aggregate: {out_csv6}')


## 6b — Multi-Video Comparison: GT vs All Three Trackers (with Post-Processing Merge)

Same as Section 6 but applies `merge_fragments()` to each tracker output before
counting fragments. Allows direct comparison of raw vs merged performance.

**Configuration knobs:**
- `N_SAMPLE_B`, `CONF_6B` — sample size and detection confidence
- `MAX_GAP` — maximum frame gap to bridge when stitching fragments
- `MAX_DIST_PCT` — maximum centroid distance (% of frame diagonal) to consider a merge

In [ ]:
import time as _time

# ── Configuration ──────────────────────────────────────────────────────────
N_SAMPLE_B   = 5      # number of video chunks to sample
CONF_6B      = 0.25   # YOLO detection confidence
MAX_GAP      = 45     # merge: max frames between fragments
MAX_DIST_PCT = 35.0   # merge: max centroid distance as % of frame diagonal
SEED_B       = 42     # set to SEED to reuse the same sample_videos list

random.seed(SEED_B)
sample_videos_b = random.sample(all_videos, min(N_SAMPLE_B, len(all_videos)))
# If seeds match, the lists will be identical
print(f'Sampled {len(sample_videos_b)} videos (conf={CONF_6B}, max_gap={MAX_GAP}, max_dist={MAX_DIST_PCT}%):')
for v in sample_videos_b: print(f'  {v.name}')

per_video_tables_6b = []

for vid in sample_videos_b:
    vname = vid.stem
    print(f'\n{"="*55}\nVideo: {vname}\n{"="*55}')

    # Ground truth (merge has no meaning here — GT tracks are already ground truth)
    df_gt_v, _ = load_gt_as_df(vid, MOT_SEQ_DIR)
    gt_frags = count_fragments(df_gt_v) if not df_gt_v.empty else {c: 0 for c in CLASSES}

    # Position-only + merge
    t0 = _time.time()
    df_pos_b = run_tracker(vid, WEIGHTS, str(_pos_cfg), conf=CONF_6B)
    df_pos_m = merge_fragments(df_pos_b, max_gap=MAX_GAP, max_dist_pct=MAX_DIST_PCT)
    t_pos_b = _time.time() - t0
    pos_frags_b = count_fragments(df_pos_m)

    # Option A + merge
    t0 = _time.time()
    df_ra_b = run_tracker(vid, WEIGHTS, str(_reid_a_cfg), conf=CONF_6B)
    df_ra_m = merge_fragments(df_ra_b, max_gap=MAX_GAP, max_dist_pct=MAX_DIST_PCT)
    t_ra_b = _time.time() - t0
    ra_frags_b = count_fragments(df_ra_m)

    # Option B + merge
    if 'REID_MODEL' in dir() and Path(REID_MODEL).exists():
        t0 = _time.time()
        df_rb_b = run_tracker_boxmot(vid, WEIGHTS,
                                     track_buffer=int(reid_b_best['track_buffer']),
                                     match_thresh=float(reid_b_best['match_thresh']),
                                     appearance_thresh=float(reid_b_best.get('appearance_thresh', 0.40)),
                                     conf=CONF_6B)
        df_rb_m = merge_fragments(df_rb_b, max_gap=MAX_GAP, max_dist_pct=MAX_DIST_PCT)
        t_rb_b = _time.time() - t0
        rb_frags_b = count_fragments(df_rb_m)
    else:
        rb_frags_b = {c: None for c in CLASSES}; t_rb_b = None
        print('  Option B skipped — ReID weights not downloaded yet')

    tbl = pd.DataFrame({
        'Ground Truth':             [gt_frags.get(c, 0)   for c in CLASSES],
        'Pos+Merge':                [pos_frags_b.get(c, 0) for c in CLASSES],
        'ReID Opt-A+Merge (Ult.)':  [ra_frags_b.get(c, 0)  for c in CLASSES],
        'ReID Opt-B+Merge (bxmt)':  [rb_frags_b.get(c)     for c in CLASSES],
    }, index=CLASSES)
    tbl.loc['TOTAL']    = tbl.sum()
    tbl.loc['Time (s)'] = [None, round(t_pos_b, 1), round(t_ra_b, 1),
                            round(t_rb_b, 1) if t_rb_b is not None else None]
    per_video_tables_6b.append((vname, tbl))

    print(f'\nFragment counts (post-merge) — {vname}:')
    display(tbl.style
               .highlight_min(axis=1, color='#c6efce', subset=pd.IndexSlice[CLASSES, ['Pos+Merge','ReID Opt-A+Merge (Ult.)','ReID Opt-B+Merge (bxmt)']])
               .format(lambda x: f'{x:.1f}' if isinstance(x, float) and not pd.isna(x) else (str(int(x)) if x is not None and not (isinstance(x, float) and pd.isna(x)) else '—')))

# ── Aggregate summary ────────────────────────────────────────────────────────
print(f'\n{"="*55}\nAGGREGATE (post-merge): Total fragments across sampled videos\n{"="*55}')
agg6b = pd.DataFrame({
    col: [sum(tbl.loc[c, col] or 0 for _, tbl in per_video_tables_6b if c in tbl.index and tbl.loc[c, col] is not None)
          for c in CLASSES]
    for col in ['Ground Truth', 'Pos+Merge', 'ReID Opt-A+Merge (Ult.)', 'ReID Opt-B+Merge (bxmt)']
}, index=CLASSES)
agg6b.loc['TOTAL'] = agg6b.sum()
display(agg6b.style.highlight_min(axis=1, color='#c6efce',
        subset=['Pos+Merge','ReID Opt-A+Merge (Ult.)','ReID Opt-B+Merge (bxmt)']))

out_csv6b = TRACKER_RUNS / 'multi_video_aggregate_merged.csv'
agg6b.to_csv(out_csv6b)
print(f'Saved aggregate: {out_csv6b}')


## 7 — Save Best Architecture Config

Writes two files:
- `configs/kartector_botsort_reentry.yaml` — tracker knobs (track_buffer, match_thresh, etc.)
- `configs/kartector_botsort_arch.json` — full inference + ReID + post-process settings

This JSON is read by notebooks 07 and 08 to reproduce the exact inference configuration.

**Best config decided:** ReID Option B (boxmot `osnet_x0_25_msmt17`), with post-process merge.

In [ ]:
all_results = []
if 'pos_results'    in dir(): all_results += pos_results
if 'reid_a_results' in dir(): all_results += reid_a_results
if 'reid_b_results' in dir(): all_results += reid_b_results

if all_results:
    df_all = pd.DataFrame(all_results)
    best   = df_all.sort_values('total_fragments').iloc[0]

    # ── Tracker YAML (Ultralytics format) ─────────────────────────────────────
    best_cfg = {
        'tracker_type': 'botsort', 'track_high_thresh': 0.25,
        'track_low_thresh': 0.10, 'new_track_thresh': 0.30,
        'track_buffer':  int(best['track_buffer']),
        'match_thresh':  float(best['match_thresh']),
        'fuse_score': True, 'gmc_method': 'sparseOptFlow',
        'proximity_thresh': 0.5,
        'appearance_thresh': float(best.get('appearance_thresh', 0.85)),
        'with_reid': bool(best.get('reid', False)),
    }
    out_cfg = REPO_ROOT / 'configs' / 'kartector_botsort_reentry.yaml'
    out_cfg.write_text(yaml.dump(best_cfg))
    print(f'Tracker YAML written to: {out_cfg}')
    print(yaml.dump(best_cfg))

    # ── Architecture JSON (inference + post-process settings) ─────────────────
    # Determine whether post-processing merge helped by comparing best raw vs merged
    # Use the fragment count from Section 6 (raw) vs 6b (merged) if both were run.
    # Fall back to True if 6b was run (it was explicitly configured).
    post_process_on = 'per_video_tables_6b' in dir() and bool(per_video_tables_6b)

    # MAX_GAP / MAX_DIST_PCT may have been set in Section 6b; fall back to defaults.
    arch_max_gap      = MAX_GAP      if 'MAX_GAP'      in dir() else 45
    arch_max_dist_pct = MAX_DIST_PCT if 'MAX_DIST_PCT' in dir() else 35.0

    # ── Determine preferred ReID option ───────────────────────────────────────
    # Compare best fragment counts: Option B < Option A < position-only
    best_pos_frags  = int(best_pos.total_fragments) if 'best_pos' in dir() else 9999
    best_a_frags    = (int(pd.DataFrame(reid_a_results).sort_values('total_fragments').iloc[0].total_fragments)
                       if 'reid_a_results' in dir() and reid_a_results else 9999)
    best_b_frags    = (int(pd.DataFrame(reid_b_results).sort_values('total_fragments').iloc[0].total_fragments)
                       if 'reid_b_results' in dir() and reid_b_results else 9999)

    if best_b_frags <= best_a_frags and best_b_frags <= best_pos_frags:
        reid_option = 'B'
    elif best_a_frags <= best_pos_frags:
        reid_option = 'A'
    else:
        reid_option = 'none'
    print(f'ReID option selected: {reid_option}  '
          f'(pos={best_pos_frags} A={best_a_frags} B={best_b_frags})')

    # Best boxmot (Option B) params — used by notebook 05 when reid_option=='B'
    if 'reid_b_results' in dir() and reid_b_results:
        _bb = pd.DataFrame(reid_b_results).sort_values('total_fragments').iloc[0]
        reid_b_params = {
            'track_buffer':      int(_bb.get('track_buffer', 30)),
            'match_thresh':      float(_bb.get('match_thresh', 0.70)),
            'appearance_thresh': float(_bb.get('appearance_thresh', 0.40)),
        }
    else:
        reid_b_params = {'track_buffer': 30, 'match_thresh': 0.70, 'appearance_thresh': 0.40}

    arch = {
        'tracker':       'botsort',
        'tracker_yaml':  str(out_cfg.relative_to(REPO_ROOT)),
        'conf':          float(CONF),
        'post_process':  post_process_on,
        'max_gap':       int(arch_max_gap),
        'max_dist_pct':  float(arch_max_dist_pct),
        'reid_option':   reid_option,      # 'A', 'B', or 'none'
        'reid_b_params': reid_b_params,    # boxmot osnet params (used when reid_option=='B')
    }
    import json as _json
    arch_out = REPO_ROOT / 'configs' / 'kartector_botsort_arch.json'
    arch_out.write_text(_json.dumps(arch, indent=2))
    print(f'Architecture config written to: {arch_out}')
    print(_json.dumps(arch, indent=2))
else:
    print('Run at least one sweep section first.')
